# Pakistan Largest E-Commerce Dataset

**Prepared by Arsalan Khan**

This notebook studies order data from Pakistan's largest e-commerce dataset. The goal is simple: understand sales, customers, products, payments, discounts, weak points, and better business choices.

The notebook uses only basic pandas. Each code cell has one or two lines. Read the note above each code cell, run the code, then read the business meaning below it.

Do not treat this file as a profit report. The dataset has sales values, but it does not give true product cost, delivery cost, warehouse cost, staff cost, return cost, or marketing cost. So we can study sales and make a simple profit plan, but we cannot calculate true profit unless those cost files are added later.

The dataset also does not have city, province, or exact customer address. So this notebook cannot tell the company which city to sell in. It can tell which month, category, product, customer group, and payment method look better or worse.

## How to use this notebook

1. Download the dataset CSV file from Kaggle.
2. Put the CSV file in the same folder as this notebook, or change the path in the next file-name cell.
3. Run the cells from top to bottom.
4. After each table, read the plain business note.

A CSV file is a table file. It is like an Excel sheet saved in a simple text format. Pandas can read it and show it as rows and columns.

## 1. Load pandas

Pandas is the only Python library used here. We use it because it can read tables, count rows, group data, and make simple summary tables.

The short name `pd` is common. It only saves typing. It means pandas.

In [ ]:
import pandas as pd

## 2. Give the file name

This line tells Python where the CSV file is.

If your file is in the same folder as this notebook, keep the simple file name. If you are using Kaggle, the path may start with `/kaggle/input/`.

In [ ]:
file_name = "Pakistan Largest Ecommerce Dataset.csv"

## 3. Read the data

This line reads the CSV file and stores it inside a table named `data`.

`low_memory=False` helps pandas read large files more calmly. It does not change the business meaning of the data.

In [ ]:
data = pd.read_csv(file_name, low_memory=False)

## 4. See the first rows

`head()` shows the first five rows. This is a quick check. It helps us see whether the file opened correctly.

Look at the column names, the dates, the product names, the prices, and the order status.

In [ ]:
data.head()

## 5. Count rows and columns

`shape` gives two numbers. The first number is the number of rows. The second number is the number of columns.

A row usually means one order item record. A column is one kind of information, such as price, date, or payment method.

In [ ]:
data.shape

## 6. See all column names

This shows the names of all columns in the file.

Column names are important because pandas uses exact spelling. For example, `Customer ID` and `customer id` are not the same for Python.

In [ ]:
data.columns

## 7. Check the basic data types

`info()` shows column names, how many values are present, and what type of data each column has.

Common types are `object` for text, `float` for decimal numbers, and `int` for whole numbers. Dates may first appear as text, so we fix them later.

In [ ]:
data.info()

# Basic cleaning

Cleaning means making the table easier to study. We do not change the story of the data. We only remove fully empty rows, remove blank extra columns, fix date columns, fix number columns, and create simple helper columns.

## 8. Remove rows that are fully empty

Some files have blank rows at the bottom. A fully empty row has no useful value in any column.

`dropna(how="all")` removes only those rows where all values are missing.

In [ ]:
clean = data.dropna(how="all")

## 9. Check the new size

After removing fully empty rows, we check the size again.

If the number of rows falls a lot, the file had many blank rows. That is common in some exported files.

In [ ]:
clean.shape

## 10. Name the blank extra columns

This dataset may contain columns named `Unnamed: 21` to `Unnamed: 25`. These are usually empty columns from the file export.

We put their names in a small list, then remove them in the next cell.

In [ ]:
empty_columns = ["Unnamed: 21", "Unnamed: 22", "Unnamed: 23", "Unnamed: 24", "Unnamed: 25"]

## 11. Remove the blank extra columns

`drop(columns=...)` removes columns.

`errors="ignore"` means: if one of these columns is not found, do not stop the notebook. This makes the notebook safer for beginners.

In [ ]:
clean = clean.drop(columns=empty_columns, errors="ignore")

## 12. Check missing values

A missing value means the cell is blank or not available.

This table counts missing values in each column. Columns with many missing values need care. We should not hide missing values without thinking.

In [ ]:
clean.isna().sum().sort_values(ascending=False)

## 13. Check duplicate rows

A duplicate row is an exact copy of another row.

If the count is high, we should ask why. Some repeated orders may be valid, but exact duplicate rows may be a file problem.

In [ ]:
clean.duplicated().sum()

## 14. Fix the order date

`created_at` is the order date. We convert it into a real date so pandas can find month, year, and weekday.

`errors="coerce"` means bad dates become missing dates instead of breaking the notebook.

In [ ]:
clean["created_at"] = pd.to_datetime(clean["created_at"], errors="coerce")

## 15. Fix the working date

`Working Date` is the date when the order was processed or worked on.

We also convert it into a real date. Later, we use it to study how many days processing took.

In [ ]:
clean["Working Date"] = pd.to_datetime(clean["Working Date"], errors="coerce")

## 16. Fix price and quantity columns

Prices and quantities must be numbers. If they stay as text, pandas cannot add them properly.

`to_numeric` turns a column into numbers. Bad values become missing values.

In [ ]:
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")
clean["qty_ordered"] = pd.to_numeric(clean["qty_ordered"], errors="coerce")

## 17. Fix total sales and discount columns

`grand_total` is the total order value in the file. `discount_amount` is the discount amount.

Both must be numbers because we will sum them, compare them, and group them.

In [ ]:
clean["grand_total"] = pd.to_numeric(clean["grand_total"], errors="coerce")
clean["discount_amount"] = pd.to_numeric(clean["discount_amount"], errors="coerce")

## 18. Fix MV column

`MV` is another money-related column in this dataset. Different notebooks explain it as merchandise value or money value.

We convert it to a number, but most business tables below use `grand_total` because it is easier for beginners to understand.

In [ ]:
clean["MV"] = pd.to_numeric(clean["MV"], errors="coerce")

## 19. Fill missing text values

For text columns, we fill missing values with the word `missing`.

This keeps rows in the analysis. It also lets us see how much business is tied to missing customer, product, category, or payment information.

In [ ]:
clean["status"] = clean["status"].fillna("missing")
clean["sku"] = clean["sku"].fillna("missing")

## 20. Fill more missing text values

We do the same for category, payment method, customer ID, and sales commission code.

This step is simple. It does not guess the missing value. It only labels it as missing.

In [ ]:
clean["category_name_1"] = clean["category_name_1"].fillna("missing")
clean["payment_method"] = clean["payment_method"].fillna("missing")

## 21. Fill customer and sales code blanks

A missing customer ID means we cannot link that order to a known customer.

A missing sales commission code means we cannot study the sales source properly for that row.

In [ ]:
clean["Customer ID"] = clean["Customer ID"].fillna("missing")
clean["sales_commission_code"] = clean["sales_commission_code"].fillna("missing")

## 22. Make date helper columns

These helper columns make time analysis easy.

`order_year` gives the year. `order_month` gives year and month together. These are useful for trend tables.

In [ ]:
clean["order_year"] = clean["created_at"].dt.year
clean["order_month"] = clean["created_at"].dt.to_period("M").astype(str)

## 23. Make weekday column

`order_weekday` gives the day name, such as Monday or Friday.

This helps answer: which day gets more orders or more sales?

In [ ]:
clean["order_weekday"] = clean["created_at"].dt.day_name()

## 24. Make gross sales and discount rate

`gross_sales` is price multiplied by quantity.

`discount_rate` shows discount as a share of gross sales. For example, 0.10 means 10%. This is only a simple estimate because some orders may have special rules.

In [ ]:
clean["gross_sales"] = clean["price"] * clean["qty_ordered"]
clean["discount_rate"] = clean["discount_amount"] / clean["gross_sales"]

## 25. Clean discount rate

Sometimes discount rate becomes missing or infinite when the gross sales value is zero or missing.

We change those bad values to zero so beginner tables do not break. In a company report, these rows should be checked separately.

In [ ]:
clean["discount_rate"] = clean["discount_rate"].replace([float("inf"), -float("inf")], 0)
clean["discount_rate"] = clean["discount_rate"].fillna(0)

## 26. Mark cancelled and refunded orders

A cancelled order is usually not good revenue. A refunded order may also reduce real income.

These flags help us separate good-looking sales from orders that caused problems.

In [ ]:
clean["is_cancelled"] = clean["status"].str.lower().str.contains("cancel", na=False)
clean["is_refund"] = clean["status"].str.lower().str.contains("refund", na=False)

## 27. Mark problem orders

Here, a problem order means it was cancelled or refunded.

This is a simple rule. A real company may add more problem statuses later, such as failed delivery or payment failed.

In [ ]:
clean["problem_order"] = clean["is_cancelled"] | clean["is_refund"]

## 28. Make a simple net sales column for planning

This column starts with `grand_total`. Then we set cancelled or refunded order sales to zero.

This is not perfect accounting. It is a safer planning number than raw sales because problem orders should not be treated as clean income.

In [ ]:
clean["net_sales_for_planning"] = clean["grand_total"]
clean.loc[clean["problem_order"], "net_sales_for_planning"] = 0

## 29. Make customer join date

`Customer Since` tells when the customer joined. We convert it into a date if pandas can understand it.

This helps us study old customers and new customers.

In [ ]:
clean["customer_join_date"] = pd.to_datetime(clean["Customer Since"], errors="coerce")

## 30. Make processing days

Processing days means the gap between order date and working date.

If this number is high, orders may be slow to process. Slow processing can hurt customer trust.

In [ ]:
clean["processing_days"] = (clean["Working Date"] - clean["created_at"]).dt.days

## 31. Simple profit planning setup

The dataset does not contain actual product cost. So this is only a planning estimate.

Here we assume the cost is 70% of net sales. That leaves 30% as estimated profit. Change `0.70` if the company has a better cost estimate.

In [ ]:
assumed_cost_rate = 0.70
clean["estimated_profit"] = clean["net_sales_for_planning"] * (1 - assumed_cost_rate)

# Analysis 1: order status count

This table counts each order status.

Business use: the company should not only chase more orders. It should also reduce cancelled, refunded, and failed orders. A high count of bad statuses means lost work, unhappy customers, and wasted delivery effort.

In [ ]:
status_count = clean["status"].value_counts()
status_count

# Analysis 2: sales by order status

This table adds sales value for each status.

Business use: if cancelled or refunded statuses have large sales value, the company may be losing money where sales look strong on the surface. First fix the reasons behind those bad statuses.

In [ ]:
sales_by_status = clean.groupby("status")["grand_total"].sum().sort_values(ascending=False)
sales_by_status

# Analysis 3: total raw sales

This gives the total value of `grand_total`.

Business use: this is the large top-line number, but do not treat it as true profit. It may include cancelled or refunded orders.

In [ ]:
total_raw_sales = clean["grand_total"].sum()
total_raw_sales

# Analysis 4: total planning sales

This gives sales after setting cancelled and refunded orders to zero.

Business use: this is a safer number for planning than raw sales. It still does not remove delivery cost, product cost, marketing cost, or staff cost.

In [ ]:
total_planning_sales = clean["net_sales_for_planning"].sum()
total_planning_sales

# Analysis 5: sales by year

This table shows sales in each year.

Business use: if one year is stronger, study what happened that year. It may be better campaigns, more products, better payment options, or more trust in online shopping.

In [ ]:
sales_by_year = clean.groupby("order_year")["net_sales_for_planning"].sum().sort_index()
sales_by_year

# Analysis 6: sales by month over time

This table shows sales month by month.

Business use: this helps planning. Strong months need more stock, more staff, and better delivery support. Weak months need offers, reminders, and better product bundles.

In [ ]:
sales_by_month = clean.groupby("order_month")["net_sales_for_planning"].sum().sort_index()
sales_by_month

# Analysis 7: best sales months

This table shows the months with the highest sales.

Business use: plan big campaigns before these months, not after them. Prepare stock and delivery capacity early.

In [ ]:
best_sales_months = sales_by_month.sort_values(ascending=False)
best_sales_months.head(12)

# Analysis 8: order count by month

This counts orders in each month.

Business use: sales can rise because of more orders or because of expensive orders. This table checks order volume, not money value.

In [ ]:
orders_by_month = clean.groupby("order_month")["item_id"].count().sort_index()
orders_by_month

# Analysis 9: average order value by month

Average order value means sales divided by number of orders.

Business use: if a month has many small orders, push bundles. If a month has fewer but expensive orders, protect service quality and avoid cancellations.

In [ ]:
monthly_average_order_value = clean.groupby("order_month")["net_sales_for_planning"].mean().sort_index()
monthly_average_order_value

# Analysis 10: sales by weekday

This shows which weekdays bring more sales.

Business use: run reminders and offers before strong days. Put more customer support on days with heavy orders.

In [ ]:
sales_by_weekday = clean.groupby("order_weekday")["net_sales_for_planning"].sum().sort_values(ascending=False)
sales_by_weekday

# Analysis 11: order count by weekday

This counts how many orders arrive on each weekday.

Business use: if one day has many orders, warehouse and support teams should be ready on that day.

In [ ]:
orders_by_weekday = clean.groupby("order_weekday")["item_id"].count().sort_values(ascending=False)
orders_by_weekday

# Analysis 12: orders by category

This shows which product categories get the most order records.

Business use: high-order categories bring traffic. Keep them visible on the homepage and make sure popular items do not go out of stock.

In [ ]:
orders_by_category = clean["category_name_1"].value_counts()
orders_by_category

# Analysis 13: sales by category

This shows which categories bring the most sales value.

Business use: categories with high sales deserve better stock planning, faster delivery, and clear product pages. But also check cancellations before investing more.

In [ ]:
sales_by_category = clean.groupby("category_name_1")["net_sales_for_planning"].sum().sort_values(ascending=False)
sales_by_category

# Analysis 14: average order value by category

This shows the average sales value per order for each category.

Business use: high average value categories need careful service. A single bad order can lose more money and more trust.

In [ ]:
aov_by_category = clean.groupby("category_name_1")["net_sales_for_planning"].mean().sort_values(ascending=False)
aov_by_category

# Analysis 15: cancellation rate by category

This shows the share of orders that were cancelled in each category.

Business use: if a category has high sales but also high cancellation, do not just sell more. First fix stock accuracy, delivery promises, product quality, or payment issues.

In [ ]:
cancel_rate_by_category = clean.groupby("category_name_1")["is_cancelled"].mean().sort_values(ascending=False)
cancel_rate_by_category

# Analysis 16: refund rate by category

This shows the share of refunded orders in each category.

Business use: refunds may point to wrong product descriptions, damaged items, poor quality, or customer expectation problems. Fix the cause before pushing that category harder.

In [ ]:
refund_rate_by_category = clean.groupby("category_name_1")["is_refund"].mean().sort_values(ascending=False)
refund_rate_by_category

# Analysis 17: average discount rate by category

This shows which categories use heavier discounts.

Business use: a category that sells only with large discounts may be weak. Check if discount is bringing new buyers or only reducing income.

In [ ]:
discount_rate_by_category = clean.groupby("category_name_1")["discount_rate"].mean().sort_values(ascending=False)
discount_rate_by_category

# Analysis 18: payment method count

This counts how many orders used each payment method.

Business use: popular payment methods should be smooth and reliable. If cash on delivery is high, the company must manage failed delivery and cancellation risk carefully.

In [ ]:
payment_method_count = clean["payment_method"].value_counts()
payment_method_count

# Analysis 19: sales by payment method

This shows how much planning sales came through each payment method.

Business use: payment methods that bring large sales should get better testing, faster issue handling, and clear instructions for customers.

In [ ]:
sales_by_payment = clean.groupby("payment_method")["net_sales_for_planning"].sum().sort_values(ascending=False)
sales_by_payment

# Analysis 20: cancellation rate by payment method

This shows which payment methods have more cancellations.

Business use: if one payment method has high cancellations, review payment steps, failed payment reasons, customer trust, and delivery confirmation rules.

In [ ]:
cancel_rate_by_payment = clean.groupby("payment_method")["is_cancelled"].mean().sort_values(ascending=False)
cancel_rate_by_payment

# Analysis 21: payment method and status table

This table shows status counts for each payment method.

Business use: it helps find where order problems sit. For example, one method may have many complete orders, while another has many cancelled orders.

In [ ]:
payment_status_table = pd.crosstab(clean["payment_method"], clean["status"])
payment_status_table

# Analysis 22: category and payment method combinations

This table shows which category and payment method pairs appear most often.

Business use: the company can design payment messages by category. For example, expensive categories may need stronger trust signs and easier payment options.

In [ ]:
category_payment_orders = clean.groupby(["category_name_1", "payment_method"])["item_id"].count().sort_values(ascending=False)
category_payment_orders.head(25)

# Analysis 23: top products by quantity

`sku` means product code. This table shows which products sold the most units.

Business use: keep these products in stock. If they are low-margin, bundle them with better-margin items.

In [ ]:
top_skus_by_quantity = clean.groupby("sku")["qty_ordered"].sum().sort_values(ascending=False)
top_skus_by_quantity.head(25)

# Analysis 24: top products by sales

This table shows which products brought the highest planning sales.

Business use: these products deserve strong product pages, clear photos, fast restocking, and careful delivery handling.

In [ ]:
top_skus_by_sales = clean.groupby("sku")["net_sales_for_planning"].sum().sort_values(ascending=False)
top_skus_by_sales.head(25)

# Analysis 25: low-selling products

This shows products with the lowest quantity sold.

Business use: low-selling products may need better photos, better price, bundles, or removal from the catalog. Do not remove before checking if the product is new or seasonal.

In [ ]:
low_skus_by_quantity = clean.groupby("sku")["qty_ordered"].sum().sort_values()
low_skus_by_quantity.head(25)

# Analysis 26: product cancellation risk

This table checks products with at least 50 order records, then shows high cancellation rates.

Business use: do not push a product harder if it often gets cancelled. First check wrong price, wrong stock, late delivery, or weak product information.

In [ ]:
sku_cancel_table = clean.groupby("sku").agg({"item_id": "count", "is_cancelled": "mean"})
sku_cancel_table[sku_cancel_table["item_id"] >= 50].sort_values("is_cancelled", ascending=False).head(25)

# Analysis 27: make price bands

A price band groups prices into simple ranges.

This helps beginners compare cheap, medium, and expensive items without reading thousands of separate prices.

In [ ]:
clean["price_band"] = pd.cut(clean["price"], bins=[0, 1000, 5000, 10000, 50000, 1000000])

# Analysis 28: order count by price band

This counts orders in each price range.

Business use: if most orders are low-price, the company may need bundles to raise order value. If many orders are high-price, service trust becomes more important.

In [ ]:
orders_by_price_band = clean.groupby("price_band")["item_id"].count()
orders_by_price_band

# Analysis 29: sales by price band

This shows which price range brings more planning sales.

Business use: the company should not assume cheap products always matter most. A smaller number of expensive orders may bring much larger sales.

In [ ]:
sales_by_price_band = clean.groupby("price_band")["net_sales_for_planning"].sum()
sales_by_price_band

# Analysis 30: make discount bands

A discount band groups discount rates into simple ranges.

This helps answer: do bigger discounts bring better sales, or do they only reduce income?

In [ ]:
clean["discount_band"] = pd.cut(clean["discount_rate"], bins=[-1, 0, 0.05, 0.10, 0.20, 1])

# Analysis 31: sales by discount band

This shows planning sales in each discount range.

Business use: if high discounts bring little sales, stop wasting discount money. If medium discounts bring strong sales, use them only on selected products and months.

In [ ]:
sales_by_discount_band = clean.groupby("discount_band")["net_sales_for_planning"].sum()
sales_by_discount_band

# Analysis 32: cancellation by discount band

This shows whether heavy discounts are linked with cancellations.

Business use: if high-discount orders cancel more, the company may be attracting weak orders or setting poor delivery promises.

In [ ]:
cancel_by_discount_band = clean.groupby("discount_band")["is_cancelled"].mean()
cancel_by_discount_band

# Analysis 33: sales with and without discount

This separates orders into discount and no-discount groups.

Business use: if discounted orders bring much lower average sales or more cancellations, the discount plan needs control.

In [ ]:
clean["has_discount"] = clean["discount_amount"] > 0
clean.groupby("has_discount")["net_sales_for_planning"].sum()

# Analysis 34: average order value with and without discount

This compares average planning sales for discounted and non-discounted orders.

Business use: discounts should not be judged by sales only. A discount is useful when it raises good orders, not when it lowers value or increases bad orders.

In [ ]:
discount_aov = clean.groupby("has_discount")["net_sales_for_planning"].mean()
discount_aov

# Analysis 35: valid customer table

Some rows have missing customer ID. We create a separate table that keeps only known customers.

Business use: customer behavior analysis should be done on known customers, not missing IDs.

In [ ]:
valid_customers = clean[clean["Customer ID"] != "missing"]

# Analysis 36: number of known customers

This counts unique known customers.

Business use: the company should track both buyer count and order count. More orders from fewer customers means repeat buying. More customers with few orders means weak retention.

In [ ]:
known_customer_count = valid_customers["Customer ID"].nunique()
known_customer_count

# Analysis 37: top customers by sales

This shows customers who brought the most planning sales.

Business use: high-value customers should get better service, early sale access, and quick complaint handling. Losing them hurts more.

In [ ]:
top_customers_by_sales = valid_customers.groupby("Customer ID")["net_sales_for_planning"].sum().sort_values(ascending=False)
top_customers_by_sales.head(25)

# Analysis 38: customer order count

This counts how many orders each known customer made.

Business use: repeat customers are cheaper to serve than always finding new buyers. The company should make repeat buying easy.

In [ ]:
customer_order_count = valid_customers.groupby("Customer ID")["item_id"].count().sort_values(ascending=False)
customer_order_count.head(25)

# Analysis 39: repeat customer check

This shows how many customers ordered once and how many ordered more than once.

Business use: if most customers buy only once, the company needs follow-up offers, better delivery, and better complaint handling.

In [ ]:
repeat_customer_check = (customer_order_count > 1).value_counts()
repeat_customer_check

# Analysis 40: customer sales summary

`describe()` gives count, average, middle value, minimum, maximum, and other simple numbers.

Business use: if the maximum is very high compared with the middle value, a few customers may be bringing a large share of sales. Protect them, but do not depend only on them.

In [ ]:
customer_sales_summary = top_customers_by_sales.describe()
customer_sales_summary

# Analysis 41: customer joining trend

This counts known customers by their joining month in `Customer Since`.

Business use: months with many new customers may be linked with strong campaigns or market events. Study those months and repeat what worked.

In [ ]:
customers_by_join_month = valid_customers.groupby("Customer Since")["Customer ID"].nunique().sort_index()
customers_by_join_month

# Analysis 42: processing days summary

This gives a simple summary of processing time.

Business use: slow processing can increase cancellations and complaints. If the average or maximum is high, check warehouse and order approval steps.

In [ ]:
processing_days_summary = clean["processing_days"].describe()
processing_days_summary

# Analysis 43: slow order flag

Here we call an order slow if processing took more than 3 days.

This is a simple rule for teaching. The company can change 3 days to its own service promise.

In [ ]:
clean["slow_order"] = clean["processing_days"] > 3

# Analysis 44: slow processing by category

This shows which categories have more slow orders.

Business use: if a category is slow, it may need better stock control, supplier coordination, or separate packing rules.

In [ ]:
slow_by_category = clean.groupby("category_name_1")["slow_order"].mean().sort_values(ascending=False)
slow_by_category

# Analysis 45: slow processing by payment method

This shows whether some payment methods are linked with slower processing.

Business use: if one payment method delays order handling, fix approval or confirmation steps.

In [ ]:
slow_by_payment = clean.groupby("payment_method")["slow_order"].mean().sort_values(ascending=False)
slow_by_payment

# Analysis 46: sales commission code count

This counts orders by sales commission code.

Business use: if these codes represent sales people, partners, or campaigns, the company can compare which sources bring orders. Missing codes must be fixed before judging people or partners.

In [ ]:
sales_code_count = clean["sales_commission_code"].value_counts()
sales_code_count.head(25)

# Analysis 47: sales by sales commission code

This shows planning sales by sales commission code.

Business use: a code with many orders but low sales may need different products. A code with fewer orders but high sales may be bringing better buyers.

In [ ]:
sales_by_sales_code = clean.groupby("sales_commission_code")["net_sales_for_planning"].sum().sort_values(ascending=False)
sales_by_sales_code.head(25)

# Analysis 48: category planning table

This table joins four useful numbers for each category: order count, planning sales, cancellation rate, and average discount rate.

Business use: this is a decision table. Good categories have strong sales, enough orders, low cancellations, and controlled discounts.

In [ ]:
category_plan = clean.groupby("category_name_1").agg({"item_id": "count", "net_sales_for_planning": "sum", "is_cancelled": "mean", "discount_rate": "mean"})
category_plan.sort_values("net_sales_for_planning", ascending=False).head(20)

# Analysis 49: product planning table

This table joins order count, planning sales, cancellation rate, and average discount rate for each product code.

Business use: products with high sales and low cancellation should get better stock support. Products with high cancellation need a fix before more marketing.

In [ ]:
sku_plan = clean.groupby("sku").agg({"item_id": "count", "net_sales_for_planning": "sum", "is_cancelled": "mean", "discount_rate": "mean"})
sku_plan.sort_values("net_sales_for_planning", ascending=False).head(25)

# Analysis 50: payment planning table

This table joins order count, planning sales, cancellation rate, and slow-order rate for each payment method.

Business use: a good payment method brings sales without creating many cancellations or delays.

In [ ]:
payment_plan = clean.groupby("payment_method").agg({"item_id": "count", "net_sales_for_planning": "sum", "is_cancelled": "mean", "slow_order": "mean"})
payment_plan.sort_values("net_sales_for_planning", ascending=False)

# Analysis 51: month planning table

This table joins order count, planning sales, cancellation rate, and slow-order rate for each month.

Business use: months with high sales and high slow-order rate need more warehouse and delivery support. Strong demand without support can create bad customer experience.

In [ ]:
month_plan = clean.groupby("order_month").agg({"item_id": "count", "net_sales_for_planning": "sum", "is_cancelled": "mean", "slow_order": "mean"})
month_plan.sort_values("net_sales_for_planning", ascending=False).head(15)

# Analysis 52: estimated profit by category

This uses the assumed cost rate from earlier. It is not true profit. It is a simple planning estimate.

Business use: after the company adds real product cost and delivery cost, this table should be rebuilt with real profit. Until then, use it only for rough planning.

In [ ]:
estimated_profit_by_category = clean.groupby("category_name_1")["estimated_profit"].sum().sort_values(ascending=False)
estimated_profit_by_category

# Analysis 53: estimated profit by product

This shows estimated profit by product code using the same simple cost assumption.

Business use: high estimated profit products should be checked for stock, delivery success, and customer complaints before scaling.

In [ ]:
estimated_profit_by_sku = clean.groupby("sku")["estimated_profit"].sum().sort_values(ascending=False)
estimated_profit_by_sku.head(25)

# Analysis 54: category-payment sales table

This table puts categories in rows and payment methods in columns.

Business use: it helps answer how customers pay in each category. The company can make better payment messages for each category.

In [ ]:
category_payment_sales = clean.pivot_table(index="category_name_1", columns="payment_method", values="net_sales_for_planning", aggfunc="sum")
category_payment_sales.head(20)

# Analysis 55: category-status table

This table puts categories in rows and order statuses in columns.

Business use: it helps find which categories have many cancelled, completed, refunded, or closed orders. This is useful before spending more on a category.

In [ ]:
category_status_table = pd.crosstab(clean["category_name_1"], clean["status"])
category_status_table.head(20)

# Final business plan from this notebook

Use the tables above in this order.

First, pick categories with high planning sales, enough orders, low cancellation rate, and controlled discount rate. These are the safest categories to grow.

Second, protect the top products by quantity and sales. Keep stock ready. Improve product pages. Make return and delivery rules clear.

Third, stop treating discounts as a habit. Compare discount bands. Keep the discount level that brings good sales without raising cancellations. Remove weak discounts that only reduce income.

Fourth, fix payment methods with high cancellations. A payment method is not good just because it is popular. It must also complete orders.

Fifth, plan by month and weekday. Strong months and strong days need stock, staff, and delivery support before demand arrives. Weak months need selected offers, not random discounts.

Sixth, use customer tables to build repeat buying. Give known repeat customers better support. Send follow-up offers after first purchase. Make complaint handling faster for high-value customers.

Seventh, do not claim city-wise strategy from this dataset. There is no city or province column. For city planning, the company must add delivery city, shipping area, or customer location data.

Eighth, do not claim true profit from this dataset alone. Add product cost, delivery cost, return cost, marketing cost, and staff cost. Then rebuild profit tables using real costs.

# Simple decision checklist

Use this checklist after running the notebook.

1. Which category has high sales and low cancellation? Sell more there.
2. Which category has high sales and high cancellation? Fix it before selling more.
3. Which product sells many units? Keep it in stock.
4. Which product has high cancellation? Check stock, price, product page, and delivery time.
5. Which payment method gives high sales and low cancellation? Make it easy to use.
6. Which payment method gives many cancellations? Fix payment and order confirmation steps.
7. Which months are strongest? Prepare campaigns, stock, and delivery early.
8. Which weekdays are strongest? Schedule support and packing staff around those days.
9. Do discounts raise good sales? Keep them.
10. Do discounts raise cancellations or weak orders? Reduce them.
11. Are repeat customers low? Build follow-up offers and better service.
12. Are many customer IDs missing? Fix customer tracking before making customer decisions.

# What data should be added next

This dataset is useful, but the company needs more columns for better decisions.

Add city, province, delivery time, delivery partner, product cost, shipping cost, return reason, customer complaint reason, ad campaign name, traffic source, and product margin.

With those columns, the company can answer deeper questions: where to sell, which delivery partner fails more, which campaign brings good customers, and which products bring real profit.